[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C19_Bayesian_ML_Course/02_mcmc/02_mcmc.ipynb)

# 02 · 马尔可夫链蒙特卡洛 MCMC（纯 numpy 从零）

目标：从零实现 **Metropolis-Hastings** 与 **Gibbs 采样**，用它们采后验，并与**真分布的解析矩 / 共轭后验**对拍；再从零实现收敛诊断 **自相关 / ESS / $\hat R$**。

路线：MH 采高斯（对拍解析矩）→ 步长扫描与接受率 → MH 采 Beta 后验（对拍共轭）→ Gibbs 采二维高斯（对拍解析）→ Gibbs 采 Normal 均值+精度（对拍）→ 自相关/ESS → $\hat R$ → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 心智模型：**MCMC 让一个粒子在参数空间游走，停留频率正比于后验密度**。正确性只需未归一化密度（归一化常数被接受率的比值约掉）；有用性靠诊断（$\hat R$、ESS）保证。

## 1 · Metropolis-Hastings 采标准高斯（对拍解析矩）

先在一个**已知答案**的目标上验证 MH 正确：目标 $\mathcal N(\mu,\sigma^2)$，我们只给采样器**未归一化**的 log 密度（去掉归一化常数），看采出的样本均值/方差是否收敛到真值。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def check_allclose(name, got, ref, atol=1e-8, rtol=1e-5):
    got = np.asarray(got, float); ref = np.asarray(ref, float)
    ok = np.allclose(got, ref, atol=atol, rtol=rtol)
    err = float(np.max(np.abs(got - ref))) if got.size else 0.0
    print(f'[{name:<32}] allclose={ok}  max|err|={err:.2e}')
    assert ok, f'{name} 不一致'
    return ok

def metropolis(log_target, x0, n_samples, step, rng, burn=0):
    '''random-walk Metropolis（对称提议）。log_target 可以是未归一化的。
       返回 (samples, accept_rate)。拒绝时『再记一次』当前点。'''
    x = np.atleast_1d(np.asarray(x0, float))
    d = x.shape[0]
    logp = log_target(x)
    samples = np.empty((n_samples, d))
    n_acc = 0
    for t in range(n_samples + burn):
        prop = x + step * rng.standard_normal(d)
        logp_prop = log_target(prop)
        # 对称提议：接受率 = min(1, exp(logp_prop - logp))
        if np.log(rng.uniform()) < logp_prop - logp:
            x, logp = prop, logp_prop
            if t >= burn: n_acc += 1
        if t >= burn:
            samples[t - burn] = x
    return samples, n_acc / n_samples

# 目标：N(2, 1.5^2)，只给未归一化 log 密度
mu_true, sigma_true = 2.0, 1.5
log_target = lambda x: -0.5 * np.sum((x - mu_true) ** 2) / sigma_true ** 2
samples, acc = metropolis(log_target, x0=[0.0], n_samples=60000, step=3.0, rng=rng, burn=2000)
print(f'接受率 = {acc:.3f}')
check_allclose('MH 样本均值 vs 真值', samples.mean(), mu_true, atol=0.05)
check_allclose('MH 样本标准差 vs 真值', samples.std(), sigma_true, atol=0.05)
print('✅ MH 只用未归一化密度，采出的矩收敛到真分布 —— 归一化常数确实不需要')

## 2 · 步长的 Goldilocks 困境

步长 $s$ 太小 → 接受率高但走得慢（强自相关）；太大 → 几乎全被拒、链卡住。扫描 $s$，看接受率与混合（用样本自相关在 lag=1 的值粗看）的权衡。一维随机游走最优接受率约 **0.44**。

In [ ]:
def lag1_autocorr(x):
    x = x.ravel() - x.mean()
    return float(np.sum(x[1:] * x[:-1]) / np.sum(x * x))

print(f"{'step':>6} {'accept':>8} {'lag1-ACF':>10}")
for s in [0.1, 0.5, 2.0, 5.0, 20.0]:
    sm, ac = metropolis(log_target, [0.0], 40000, s, np.random.default_rng(1), burn=2000)
    print(f'{s:>6.1f} {ac:>8.3f} {lag1_autocorr(sm):>10.3f}')
# 太小的步长 -> 接受率接近 1 但自相关高；太大 -> 接受率极低
sm_small, ac_small = metropolis(log_target, [0.0], 40000, 0.1, np.random.default_rng(1), burn=2000)
sm_big, ac_big = metropolis(log_target, [0.0], 40000, 20.0, np.random.default_rng(1), burn=2000)
assert ac_small > 0.9, '极小步长 -> 接受率接近 1'
assert ac_big < 0.2, '极大步长 -> 接受率很低'
assert lag1_autocorr(sm_small) > lag1_autocorr(sm_big) or ac_big < 0.1
print('✅ 步长权衡：太小(高接受/慢混合) vs 太大(低接受/卡住)，最优在中间')

## 3 · MH 采 Beta-Binomial 后验（对拍模块 01 的共轭解）

现在采一个**真实后验**：Beta(2,2) 先验 + (12 次 9 正) → 共轭后验 Beta(11,5)。我们只给 MH **未归一化** log 后验（log 似然 + log 先验，且不含归一化常数），对拍共轭后验的解析均值/方差。注意 θ∈(0,1)，越界提议要返回 -inf。

In [ ]:
from math import lgamma
a0, b0, n, k = 2.0, 2.0, 12, 9
a_post, b_post = a0 + k, b0 + (n - k)   # 共轭后验 Beta(11,5)

def log_post_beta(theta):
    th = float(theta[0])
    if not (0.0 < th < 1.0):
        return -np.inf                  # 越界：密度为 0 -> log 为 -inf -> 必拒
    # log 似然(∝) + log 先验(∝), 都不含归一化常数
    return (k + a0 - 1) * np.log(th) + (n - k + b0 - 1) * np.log1p(-th)

sm, acc = metropolis(log_post_beta, [0.5], 80000, step=0.15, rng=np.random.default_rng(2), burn=3000)
ana_mean = a_post / (a_post + b_post)
ana_var = a_post * b_post / ((a_post + b_post) ** 2 * (a_post + b_post + 1))
print(f'接受率={acc:.3f}, 共轭后验 Beta({a_post:.0f},{b_post:.0f})')
check_allclose('MH 后验均值 vs 共轭解析', sm.mean(), ana_mean, atol=5e-3)
check_allclose('MH 后验方差 vs 共轭解析', sm.var(), ana_var, atol=2e-3)
print('✅ MH 采出的后验 = 模块 01 的共轭闭式后验（数值对拍解析）')

## 4 · Gibbs 采二维相关高斯（对拍解析）

二维高斯 $\mathcal N(0,\Sigma)$，$\Sigma=\begin{psmallmatrix}1&\rho\\\rho&1\end{psmallmatrix}$。完全条件有闭式：$x_1\mid x_2\sim\mathcal N(\rho x_2,\ 1-\rho^2)$（反之亦然）。Gibbs 轮流采这两个一维高斯，**接受率恒为 1**。对拍解析协方差。

In [ ]:
def gibbs_bivariate_normal(rho, n_samples, rng, burn=1000):
    '''Gibbs 采 N(0, [[1,rho],[rho,1]])。完全条件: x_i|x_j ~ N(rho*x_j, 1-rho^2)。'''
    x1, x2 = 0.0, 0.0
    cond_sd = np.sqrt(1 - rho ** 2)
    out = np.empty((n_samples, 2))
    for t in range(n_samples + burn):
        x1 = rng.normal(rho * x2, cond_sd)   # x1 | x2
        x2 = rng.normal(rho * x1, cond_sd)   # x2 | x1（用刚更新的 x1）
        if t >= burn:
            out[t - burn] = (x1, x2)
    return out

rho = 0.8
S = gibbs_bivariate_normal(rho, 100000, np.random.default_rng(3), burn=2000)
cov = np.cov(S.T)
print('样本协方差:\n', np.round(cov, 3))
true_cov = np.array([[1.0, rho], [rho, 1.0]])
check_allclose('Gibbs 协方差 vs 解析', cov, true_cov, atol=0.03)
print('✅ Gibbs（接受率恒为 1）采出的二维高斯协方差 = 解析 Σ')

## 5 · Gibbs 联合采 Normal 的均值 μ 与精度 τ（对拍共轭）

数据 $x_i\sim\mathcal N(\mu,1/\tau)$，均值方差**都未知**。用半共轭先验 $\mu\sim\mathcal N(\mu_0,1/\tau_0)$、$\tau\sim\mathrm{Gamma}(\alpha_0,\beta_0)$。完全条件都是标准分布：
- $\mu\mid\tau,x\sim\mathcal N$（精度加权，模块 01 的公式，精度用 $\tau$）；
- $\tau\mid\mu,x\sim\mathrm{Gamma}(\alpha_0+n/2,\ \beta_0+\tfrac12\sum(x_i-\mu)^2)$。

对拍：固定 τ 在真值附近时，μ 的后验均值应与解析精度加权一致；整体后验均值应接近数据均值。

In [ ]:
def gibbs_normal_mu_tau(data, mu0, tau0, alpha0, beta0, n_samples, rng, burn=1000):
    x = np.asarray(data, float); n = len(x); xbar = x.mean()
    mu, tau = xbar, 1.0
    out = np.empty((n_samples, 2))
    for t in range(n_samples + burn):
        # mu | tau, x  ~ N(精度加权均值, 1/后验精度)；数据精度 = n*tau
        prec = tau0 + n * tau
        mean = (tau0 * mu0 + tau * n * xbar) / prec
        mu = rng.normal(mean, np.sqrt(1.0 / prec))
        # tau | mu, x  ~ Gamma(alpha0 + n/2, beta0 + 0.5*Σ(x-mu)^2)
        shape = alpha0 + n / 2.0
        rate = beta0 + 0.5 * np.sum((x - mu) ** 2)
        tau = rng.gamma(shape, 1.0 / rate)     # numpy gamma 用 scale=1/rate
        if t >= burn:
            out[t - burn] = (mu, tau)
    return out

data = rng.normal(5.0, 2.0, size=50)        # 真 mu=5, 真 sigma=2 (tau=0.25)
S = gibbs_normal_mu_tau(data, mu0=0.0, tau0=0.01, alpha0=2.0, beta0=2.0,
                        n_samples=60000, rng=np.random.default_rng(4), burn=3000)
mu_s, tau_s = S[:, 0], S[:, 1]
print(f'后验 E[mu]={mu_s.mean():.3f} (数据均值={data.mean():.3f}, 真值 5)')
print(f'后验 E[tau]={tau_s.mean():.3f} (真 tau=0.25, 即 sigma≈{1/np.sqrt(tau_s.mean()):.2f})')
check_allclose('Gibbs E[mu] vs 数据均值', mu_s.mean(), data.mean(), atol=0.1)
assert abs(1/np.sqrt(tau_s.mean()) - 2.0) < 0.4, '后验 sigma 应接近真值 2'
print('✅ Gibbs 联合采 (mu, tau)：完全条件都是标准分布，对拍数据/真值一致')

## 6 · 收敛诊断：自相关、ESS、$\hat R$（从零实现）

- **ESS** $=N/(1+2\sum_k\rho_k)$，把相关样本折算成『等效独立样本』；
- **$\hat R$**：多链的 $\sqrt{\widehat{\mathrm{Var}}^+/W}$，收敛时 →1。

在已知答案的高斯目标上验证：独立样本 ESS≈N、$\hat R$≈1；高自相关链 ESS≪N。

In [ ]:
def autocorr(x, max_lag=None):
    x = np.asarray(x, float).ravel(); x = x - x.mean()
    n = len(x); max_lag = max_lag or n // 4
    var = np.sum(x * x)
    return np.array([np.sum(x[k:] * x[:n-k]) / var for k in range(max_lag)])

def ess(x):
    '''有效样本量 N/(1+2 Σρ_k)，按 Geyer 初始正序列截断求和。'''
    x = np.asarray(x, float).ravel(); n = len(x)
    rho = autocorr(x, max_lag=min(n // 4, 2000))
    s = 0.0
    for k in range(1, len(rho)):
        if rho[k] < 0:           # 截断：自相关首次变负即停（避免噪声累积）
            break
        s += rho[k]
    return n / (1.0 + 2.0 * s)

def r_hat(chains):
    '''chains: (m, n) m 条链各 n 个样本。Gelman-Rubin R-hat。'''
    chains = np.asarray(chains, float)
    m, n = chains.shape
    chain_means = chains.mean(axis=1)
    W = chains.var(axis=1, ddof=1).mean()              # 链内方差
    B = n * chain_means.var(ddof=1)                    # 链间方差
    var_plus = (n - 1) / n * W + B / n
    return np.sqrt(var_plus / W)

# 独立同分布样本：ESS≈N, R-hat≈1
iid = rng.standard_normal(10000)
print(f'iid 样本: ESS={ess(iid):.0f} / N=10000 (应接近 N)')
assert ess(iid) > 7000, '独立样本 ESS 应接近 N'

# 高自相关链（来自小步长 MH）：ESS 远小于 N
slow, _ = metropolis(log_target, [0.0], 10000, step=0.1, rng=np.random.default_rng(5), burn=1000)
print(f'小步长 MH 链: ESS={ess(slow):.0f} / N=10000 (应远小于 N)')
assert ess(slow) < 2000, '高自相关链 ESS 应远小于 N'

# R-hat: 4 条收敛的链 -> ≈1
chains = np.array([metropolis(log_target, [rng.normal(0,5)], 5000, 3.0,
                              np.random.default_rng(10+i), burn=1000)[0].ravel() for i in range(4)])
rh = r_hat(chains)
print(f'4 条收敛链 R-hat={rh:.4f} (应 <1.01)')
assert rh < 1.02, '收敛的链 R-hat 应接近 1'
print('✅ 诊断三件套：ESS 抓自相关、R-hat 抓多链一致性')

---
## ✏️ 练习 1：实现 Metropolis 接受步

实现 `mh_accept(logp_current, logp_proposal, rng)`：给定当前与候选的**未归一化 log 密度**，按对称提议的 Metropolis 规则返回是否接受（bool）。这是 MH 的心脏。

（提示：接受概率 $\min(1,e^{\Delta})$，$\Delta=\log p_{\text{prop}}-\log p_{\text{cur}}$；在 log 空间比较 $\log u<\Delta$ 避免溢出。）

In [ ]:
def mh_accept(logp_current, logp_proposal, rng):
    # TODO: 返回 True/False。候选更好(Δ>=0)必接受；更差按 exp(Δ) 概率接受。
    #       用 log(u) < Δ 的形式（u~Uniform(0,1)）以保数值稳定。
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
rng_t = np.random.default_rng(0)
# 候选严格更好 -> 必接受
assert mh_accept(-10.0, -1.0, rng_t) == True, '更高密度必接受'
# 候选极差 -> 几乎必拒
acc_far = [mh_accept(0.0, -50.0, rng_t) for _ in range(1000)]
assert sum(acc_far) == 0, '密度低 50 个 log 单位应几乎全拒'
# 相等 -> Δ=0 -> 必接受 (min(1,1)=1)
assert mh_accept(-3.0, -3.0, rng_t) == True
# 统计接受频率 ≈ exp(Δ)=exp(-1)≈0.368
freq = np.mean([mh_accept(0.0, -1.0, rng_t) for _ in range(50000)])
assert abs(freq - np.exp(-1)) < 0.02, f'接受频率应≈exp(-1)=0.368, 得 {freq:.3f}'
print('✅ 练习 1 通过：Metropolis 接受步 = min(1, exp(Δ)) 的正确实现')

## ✏️ 练习 2：推导并实现 Gibbs 完全条件

对二维高斯 $\mathcal N(0,\Sigma)$、$\Sigma=\begin{psmallmatrix}1&\rho\\\rho&1\end{psmallmatrix}$，实现完全条件采样 `cond_sample(other, rho, rng)`：返回 $x_i\mid x_j=\text{other}$ 的一个样本。

（提示：$x_i\mid x_j\sim\mathcal N(\rho\,x_j,\ 1-\rho^2)$。）

In [ ]:
def cond_sample(other, rho, rng):
    # TODO: 返回 N(rho*other, 1-rho^2) 的一个样本
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
rng_t = np.random.default_rng(1)
rho = 0.6
# 给定 other 固定，大量采样的均值≈rho*other、方差≈1-rho^2
smp = np.array([cond_sample(2.0, rho, rng_t) for _ in range(200000)])
assert abs(smp.mean() - rho*2.0) < 0.02, '条件均值应为 rho*other'
assert abs(smp.var() - (1-rho**2)) < 0.02, '条件方差应为 1-rho^2'
# 用它做一轮 Gibbs，协方差应趋于 Σ
x1=x2=0.0; out=[]
for _ in range(60000):
    x1 = cond_sample(x2, rho, rng_t); x2 = cond_sample(x1, rho, rng_t); out.append((x1,x2))
cov = np.cov(np.array(out)[2000:].T)
assert abs(cov[0,1] - rho) < 0.03, 'Gibbs 采样的相关应≈rho'
print('✅ 练习 2 通过：完全条件 N(rho·other, 1-rho²) 推导与采样正确')

## ✏️ 练习 3：实现 R-hat

实现 `r_hat(chains)`：输入 `(m, n)` 的 m 条链各 n 个样本，返回 Gelman-Rubin $\hat R$。

（提示：链内方差 $W=$ 各链方差的平均；链间方差 $B=n\cdot\mathrm{Var}(\text{各链均值})$；$\widehat{\mathrm{Var}}^+=\frac{n-1}{n}W+\frac1n B$；$\hat R=\sqrt{\widehat{\mathrm{Var}}^+/W}$。）

In [ ]:
def r_hat(chains):
    # TODO: 见提示，注意方差用 ddof=1（样本方差）
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rng_t = np.random.default_rng(2)
# 4 条来自同一分布的独立链 -> R-hat≈1
conv = rng_t.standard_normal((4, 5000))
rh_conv = r_hat(conv)
assert abs(rh_conv - 1.0) < 0.02, f'收敛链 R-hat≈1, 得 {rh_conv:.4f}'
# 4 条均值差很大的链（未收敛/困在不同处）-> R-hat 明显 >1
diverged = rng_t.standard_normal((4, 5000)) + np.array([[0],[5],[10],[15]])
rh_div = r_hat(diverged)
assert rh_div > 1.5, f'未收敛链 R-hat 应明显>1, 得 {rh_div:.3f}'
print(f'✅ 练习 3 通过：收敛 R-hat={rh_conv:.3f}, 未收敛 R-hat={rh_div:.2f}')

## ✏️ 练习 4：有效样本量 ESS

实现 `ess(x)`：用自相关 $\rho_k$ 计算 $\mathrm{ESS}=N/(1+2\sum_k\rho_k)$，求和用 **Geyer 初始正序列**截断（自相关首次变负即停）。

In [ ]:
def autocorr(x, max_lag=None):
    x = np.asarray(x, float).ravel(); x = x - x.mean()
    n = len(x); max_lag = max_lag or n // 4
    var = np.sum(x * x)
    return np.array([np.sum(x[k:] * x[:n-k]) / var for k in range(max_lag)])

def ess(x):
    # TODO: rho = autocorr(x); 从 k=1 累加 rho[k] 直到 rho[k]<0 截断;
    #       返回 n / (1 + 2*sum)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rng_t = np.random.default_rng(3)
# 独立样本: ESS≈N
iid = rng_t.standard_normal(8000)
assert ess(iid) > 5500, 'iid 的 ESS 应接近 N'
# 强正相关序列（AR(1), phi=0.9）: ESS≪N
phi = 0.9; ar = np.zeros(8000); e = rng_t.standard_normal(8000)
for t in range(1, 8000): ar[t] = phi*ar[t-1] + e[t]
ess_ar = ess(ar)
assert ess_ar < 1500, f'AR(1) phi=0.9 的 ESS 应远小于 N, 得 {ess_ar:.0f}'
# 理论: AR(1) 的 ESS≈N*(1-phi)/(1+phi)=N/19≈421
print(f'✅ 练习 4 通过：iid ESS≈{ess(iid):.0f}, AR(1) ESS≈{ess_ar:.0f} (理论≈{8000*(1-phi)/(1+phi):.0f})')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def mh_accept(logp_current, logp_proposal, rng):
    delta = logp_proposal - logp_current
    if delta >= 0:
        return True
    return np.log(rng.uniform()) < delta

In [ ]:
# 练习 2 参考答案
def cond_sample(other, rho, rng):
    return rng.normal(rho * other, np.sqrt(1 - rho ** 2))

In [ ]:
# 练习 3 参考答案
def r_hat(chains):
    chains = np.asarray(chains, float)
    m, n = chains.shape
    W = chains.var(axis=1, ddof=1).mean()
    B = n * chains.mean(axis=1).var(ddof=1)
    var_plus = (n - 1) / n * W + B / n
    return np.sqrt(var_plus / W)

In [ ]:
# 练习 4 参考答案
def ess(x):
    x = np.asarray(x, float).ravel(); n = len(x)
    rho = autocorr(x, max_lag=min(n // 4, 2000))
    s = 0.0
    for k in range(1, len(rho)):
        if rho[k] < 0:
            break
        s += rho[k]
    return n / (1.0 + 2.0 * s)

---
## 🧪 真实数据胶囊：logistic 回归的贝叶斯后验（MH 采样）

真实任务：用 MH 采一个**贝叶斯 logistic 回归**的后验——没有共轭、没有闭式解，正是 MCMC 的主场。数据用经典的合成可分数据（模拟真实二分类）。模型：$y_i\sim\mathrm{Bernoulli}(\sigma(w^\top x_i))$，权重先验 $w\sim\mathcal N(0,\tau^2 I)$（即 L2 正则）。

我们用 MH 采 $w$ 的后验，验证：① 后验均值给出合理的分类（训练精度高）；② 后验提供了权重的**不确定性**（频率派点估计给不了）。带 try/except——本胶囊不依赖任何外部数据，纯 numpy 合成。

In [ ]:
# 合成二分类数据（真实可分结构）
rng_c = np.random.default_rng(42)
N, D = 200, 2
w_true = np.array([2.0, -1.5])
X = rng_c.standard_normal((N, D))
logits = X @ w_true
y = (rng_c.uniform(size=N) < 1/(1+np.exp(-logits))).astype(float)

def sigmoid(z):
    return np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))  # 稳定 sigmoid

def log_post_logreg(w, tau=3.0):
    w = np.asarray(w, float)
    z = X @ w
    # log 似然: Σ [y*z - log(1+e^z)] （数值稳定的 log(1+e^z)=softplus）
    softplus = np.maximum(z, 0) + np.log1p(np.exp(-np.abs(z)))
    loglik = np.sum(y * z - softplus)
    logprior = -0.5 * np.sum(w ** 2) / tau ** 2   # N(0, tau^2 I)
    return loglik + logprior

# MH 采后验（2 维, 用前面的 metropolis）
W, acc = metropolis(log_post_logreg, [0.0, 0.0], 80000, step=0.25,
                    rng=np.random.default_rng(7), burn=5000)
w_post_mean = W.mean(axis=0)
w_post_std = W.std(axis=0)
print(f'接受率={acc:.3f}')
print(f'后验均值 w = {np.round(w_post_mean,2)} (真值 {w_true})')
print(f'后验标准差 w = {np.round(w_post_std,2)}  <- 权重的不确定性!')

# 用后验均值做分类，看训练精度
pred = (sigmoid(X @ w_post_mean) > 0.5).astype(float)
acc_clf = (pred == y).mean()
print(f'后验均值分类训练精度 = {acc_clf:.3f}')
assert acc_clf > 0.8, '可分数据上应有高精度'
assert np.all(np.sign(w_post_mean) == np.sign(w_true)), '权重符号应与真值一致'
assert np.all(w_post_std > 0.05), '后验应给出非零的权重不确定性'
print('✅ 胶囊验证：MH 采出非共轭 logistic 回归后验，给出分类 + 权重不确定性')

**🧪 胶囊练习**：实现 `bayes_logreg_predict_proba(W_samples, x_new)`：用**后验预测**给新点 `x_new` 的正类概率——对后验样本的预测取平均 $\frac1S\sum_s\sigma(w_s^\top x_{\text{new}})$，而非只用后验均值。这把权重不确定性传播进了预测概率（模块 01 后验预测的 MCMC 版）。

In [ ]:
def bayes_logreg_predict_proba(W_samples, x_new):
    # TODO: 对每个后验样本 w_s 算 sigmoid(w_s @ x_new)，再对所有样本取平均
    raise NotImplementedError

In [ ]:
# 自测
x_new = np.array([1.0, 1.0])
prob = bayes_logreg_predict_proba(W, x_new)
assert 0.0 <= prob <= 1.0, '概率应在 [0,1]'
# 后验预测概率 应介于 0/1 之间且接近 plug-in 但不必相等
plugin = sigmoid(W.mean(axis=0) @ x_new)
assert abs(prob - plugin) < 0.15, '后验预测应在 plug-in 附近（但因非线性不相等）'
# 远离决策边界的点 -> 概率接近极端
prob_far = bayes_logreg_predict_proba(W, np.array([5.0, -5.0]))
assert prob_far > 0.9, '强正类方向应给高概率'
print(f'✅ 胶囊练习通过：后验预测正类概率={prob:.3f} (plug-in={plugin:.3f})')

In [ ]:
# 📖 胶囊参考答案
def bayes_logreg_predict_proba(W_samples, x_new):
    z = W_samples @ np.asarray(x_new, float)   # (S,) 每个后验样本的 logit
    return float(np.mean(sigmoid(z)))          # 对后验样本平均 = 后验预测

### 小结
- **MCMC** 构造一条以后验为平稳分布的马尔可夫链，停留频率正比于后验密度；**只需未归一化密度**（归一化常数被接受率比值约掉）。
- **细致平衡** $\pi(\theta)T(\theta\to\theta')=\pi(\theta')T(\theta'\to\theta)$ 是正确性的局部充分条件；MH 接受率正是它的解。
- **Metropolis-Hastings**：提议+接受/拒绝，对称提议时 $\alpha=\min(1,e^\Delta)$；步长有 Goldilocks 困境（最优接受率 ~0.234/0.44）。
- **Gibbs**：轮流采**完全条件**，接受率恒为 1，分层共轭模型主力；强相关时混合慢。
- **诊断不可省**：自相关看混合、**ESS** 才是精度货币（$N/(1+2\sum\rho_k)$）、**$\hat R$**（多链）抓困在不同峰；未收敛的链会**沉默地给错误后验**。
- HMC/NUTS 用梯度大幅改进提议（Stan/PyMC 默认），但正确性与诊断与本模块一脉相承。

**下一站**：**模块 03 · 变分推断** —— 不再采样，而是把推断变成优化一个近似分布，用偏差换速度。